# Lesson 2.1 — From Python Collections to NumPy and Pandas

## Exploring a fictional appointments dataset

In this lesson, we begin **exploratory data analysis (EDA)** by turning familiar Python data structures into NumPy arrays and pandas tables. We will use a small, entirely fictional clinic-administration dataset.

> **Privacy note:** The records in this notebook are invented for learning. They contain no real patient information and must not be used for clinical decisions.

## Learning objectives

By the end of this notebook, you will be able to:

- Explain the difference between an observation, a feature, a value, and a data type.
- Create and inspect one- and two-dimensional NumPy arrays.
- Use vectorized NumPy operations instead of repeating arithmetic in a loop.
- Create pandas `Series` and `DataFrame` objects from Python collections.
- Save a DataFrame to CSV and read it back.
- Inspect a dataset with `head()`, `tail()`, `sample()`, `shape`, `columns`, `dtypes`, `info()`, and `describe()`.
- Build a simple data inventory before cleaning or analysing a dataset.
- Separate an observation from a possible explanation.

## Lesson roadmap

| Part | Focus |
|---|---|
| 1 | Why EDA begins with inspection |
| 2 | NumPy arrays and vectorized calculations |
| 3 | pandas Series and DataFrames |
| 4 | Loading and inspecting tabular data |
| 5 | Building a data inventory |
| 6 | Practice and knowledge check |

**Suggested working pattern:** predict the result, run the cell, explain the output, and then change one thing.

## 1. Why inspect before analysing?

> Imagine that a clinic administrator sends us an appointments file and asks, “What does this data tell us?” It is tempting to calculate an average immediately. A responsible analyst starts with more basic questions: What does one row represent? What are the columns? Which values are missing? Are numbers really stored as numbers? Could an ID be mistaken for a measurement?

EDA is the process of becoming familiar with data before drawing conclusions or building a machine-learning model. In this first lesson, our goal is **understanding the schema**, not cleaning every issue.

### Four useful terms

- **Observation:** One recorded case or event, usually represented by a row.
- **Feature:** A recorded characteristic, usually represented by a column.
- **Value:** The entry where a row and column meet.
- **Schema:** The dataset's structure: column names, meanings, data types, and rules.

### Think first

Suppose each row records one scheduled appointment.

- What could be a useful identifier?
- Which fields might be numerical?
- Which fields might be categories?
- Why should an appointment ID not be treated as a measurement, even if it contains digits?

Write brief answers here before continuing:

- **Likely identifier:** 
- **Likely numerical fields:** 
- **Likely categorical fields:** 
- **Reason an ID is not a measurement:**

## 2. Setup

NumPy is commonly imported as `np`, and pandas is commonly imported as `pd`. These aliases are conventions that keep later code concise.

If an import fails, install the packages in the environment used by this notebook:

```text
python -m pip install numpy pandas
```

Then restart the kernel and run the notebook again.

In [1]:
import numpy as np
import pandas as pd

print("NumPy version:", np.__version__)
print("pandas version:", pd.__version__)

NumPy version: 2.2.4
pandas version: 2.2.3


## 3. NumPy arrays

A regular Python list can store a sequence of waiting times. NumPy turns that sequence into an `ndarray`, which supports concise numerical operations across many values.

In [2]:
# Waiting times, in minutes, for six fictional appointments
wait_times_list = [12, 25, 8, 40, 15, 18]

print(wait_times_list)
print(type(wait_times_list))

[12, 25, 8, 40, 15, 18]
<class 'list'>


### A calculation without NumPy

Here is a familiar loop. It adds five minutes to each waiting time and stores the results in a new list.

In [3]:
adjusted_wait_times = []

for minutes in wait_times_list:
    adjusted_wait_times.append(minutes + 5)

print(adjusted_wait_times)

[17, 30, 13, 45, 20, 23]


### The same calculation with NumPy

> The calculation above works, but we had to describe each repetition. With a NumPy array, we express the operation we want—add five—and NumPy applies it element by element. This is called a **vectorized operation**.

Before running the next cell, predict its output.

In [4]:
wait_times = np.array(wait_times_list)

adjusted_wait_times_np = wait_times + 5

print("Original:", wait_times)
print("Adjusted:", adjusted_wait_times_np)
print("Object type:", type(wait_times))

Original: [12 25  8 40 15 18]
Adjusted: [17 30 13 45 20 23]
Object type: <class 'numpy.ndarray'>


### Shape and data type

- `shape` describes the array's dimensions.
- `dtype` describes the kind of values stored in the array.
- `ndim` reports the number of dimensions.
- `size` reports the total number of values.

Notice that these are **attributes**, so they do not need parentheses.

In [19]:
print("Shape:", wait_times.shape)               #shape is a tuple that indicates the size of the array in each dimension
print("Data type:", wait_times.dtype)           #dtype indicates the data type of the elements in the array
print("Dimensions:", wait_times.ndim)           #ndim indicates the number of dimensions of the array
print("Number of values:", wait_times.size)     #size indicates the total number of elements in the array


Shape: (6,)
Data type: int64
Dimensions: 1
Number of values: 6


### Numerical summaries

NumPy provides functions for common summaries. An average can be useful, but it should never be interpreted without understanding what values were included and whether the data is complete.

In [20]:
print("Mean wait:", np.mean(wait_times))                       #remember: the calculation of mean returns the average value of the elements in the array
print("Median wait:", np.median(wait_times))                   #remember: The median is the middle value in a dataset when all the numbers are sorted in order from smallest to largest
print("Minimum wait:", np.min(wait_times))                     #remember: The minimum is the smallest value in the array
print("Maximum wait:", np.max(wait_times))                     #remember: The maximum is the largest value in the array
print("Standard deviation:", np.std(wait_times).round(2))      #remember: The standard deviation measures the amount of variation or dispersion of a set of values

Mean wait: 19.666666666666668
Median wait: 16.5
Minimum wait: 8
Maximum wait: 40
Standard deviation: 10.5


### Two-dimensional arrays

The array below records the number of appointments for three fictional departments over four weeks. Rows represent departments and columns represent weeks.

In [21]:
weekly_appointments = np.array([
    [32, 35, 31, 38],  # General Practice
    [18, 20, 22, 19],  # Physiotherapy
    [12, 14, 11, 15],  # Dermatology
])

print(weekly_appointments)
print("Shape:", weekly_appointments.shape)
print("Dimensions:", weekly_appointments.ndim)

[[32 35 31 38]
 [18 20 22 19]
 [12 14 11 15]]
Shape: (3, 4)
Dimensions: 2


### Understanding `axis`

For this array:

- `axis=0` summarises **down the rows**, producing one result for each week.
- `axis=1` summarises **across the columns**, producing one result for each department.

Read the shape `(3, 4)` as “3 rows and 4 columns.”

In [22]:
appointments_per_week = weekly_appointments.sum(axis=0)           #axis=0 means we are summing along the rows (index 0, index 1 etc), which gives us the total number of appointments for each week across all departments.
appointments_per_department = weekly_appointments.sum(axis=1)     #axis=1 means we are summing along the columns, which gives us the total number of appointments for each department across all weeks.

print("Totals for each week:", appointments_per_week)
print("Totals for each department:", appointments_per_department)

Totals for each week: [62 69 64 72]
Totals for each department: [136  79  52]


### Checkpoint: NumPy

1. What is the difference between `shape` and `size`?
2. Why might `wait_times + 5` be clearer than writing a loop?
3. In the two-dimensional array, what does each row represent?
4. What would `weekly_appointments.mean(axis=1)` calculate?

In [23]:
# Try the final checkpoint calculation.
average_per_department = weekly_appointments.mean(axis=1)
average_per_department

array([34.  , 19.75, 13.  ])

## 4. From Python records to pandas

A list of dictionaries is a natural way to represent several records in Python. Each dictionary below represents one fictional appointment. The dictionary keys will become column names.

The dataset deliberately contains a small number of missing values. We will notice them today and clean them in Lesson 2.2.

In [24]:
appointments = [
    {
        "appointment_id": "APT-1001",
        "booking_date": "2026-09-01",
        "appointment_date": "2026-09-05",
        "department": "General Practice",
        "appointment_type": "Initial",
        "wait_minutes": 12,
        "scheduled_minutes": 20,
        "status": "Completed",
        "reminder_sent": True,
        "booking_channel": "Phone",
    },
    {
        "appointment_id": "APT-1002",
        "booking_date": "2026-09-01",
        "appointment_date": "2026-09-06",
        "department": "Physiotherapy",
        "appointment_type": "Follow-up",
        "wait_minutes": 25,
        "scheduled_minutes": 30,
        "status": "Completed",
        "reminder_sent": True,
        "booking_channel": "Online",
    },
    {
        "appointment_id": "APT-1003",
        "booking_date": "2026-09-02",
        "appointment_date": "2026-09-06",
        "department": "Dermatology",
        "appointment_type": "Initial",
        "wait_minutes": 8,
        "scheduled_minutes": 20,
        "status": "Completed",
        "reminder_sent": False,
        "booking_channel": "Phone",
    },
    {
        "appointment_id": "APT-1004",
        "booking_date": "2026-09-02",
        "appointment_date": "2026-09-07",
        "department": "General Practice",
        "appointment_type": "Follow-up",
        "wait_minutes": None,
        "scheduled_minutes": 15,
        "status": "Cancelled",
        "reminder_sent": True,
        "booking_channel": "Online",
    },
    {
        "appointment_id": "APT-1005",
        "booking_date": "2026-09-03",
        "appointment_date": "2026-09-08",
        "department": "Physiotherapy",
        "appointment_type": "Initial",
        "wait_minutes": 40,
        "scheduled_minutes": 45,
        "status": "Completed",
        "reminder_sent": True,
        "booking_channel": "Reception",
    },
    {
        "appointment_id": "APT-1006",
        "booking_date": "2026-09-03",
        "appointment_date": "2026-09-08",
        "department": "General Practice",
        "appointment_type": "Initial",
        "wait_minutes": 15,
        "scheduled_minutes": 20,
        "status": "Missed",
        "reminder_sent": False,
        "booking_channel": "Phone",
    },
    {
        "appointment_id": "APT-1007",
        "booking_date": "2026-09-04",
        "appointment_date": "2026-09-09",
        "department": "Dermatology",
        "appointment_type": "Follow-up",
        "wait_minutes": 18,
        "scheduled_minutes": 15,
        "status": "Completed",
        "reminder_sent": True,
        "booking_channel": "Online",
    },
    {
        "appointment_id": "APT-1008",
        "booking_date": "2026-09-04",
        "appointment_date": "2026-09-10",
        "department": "General Practice",
        "appointment_type": "Initial",
        "wait_minutes": 6,
        "scheduled_minutes": 20,
        "status": "Completed",
        "reminder_sent": True,
        "booking_channel": "Reception",
    },
    {
        "appointment_id": "APT-1009",
        "booking_date": "2026-09-05",
        "appointment_date": "2026-09-10",
        "department": "Physiotherapy",
        "appointment_type": "Follow-up",
        "wait_minutes": 22,
        "scheduled_minutes": 30,
        "status": "Completed",
        "reminder_sent": True,
        "booking_channel": "Online",
    },
    {
        "appointment_id": "APT-1010",
        "booking_date": "2026-09-05",
        "appointment_date": "2026-09-11",
        "department": "Dermatology",
        "appointment_type": "Initial",
        "wait_minutes": None,
        "scheduled_minutes": 20,
        "status": "Cancelled",
        "reminder_sent": False,
        "booking_channel": "Phone",
    },
    {
        "appointment_id": "APT-1011",
        "booking_date": "2026-09-06",
        "appointment_date": "2026-09-12",
        "department": "General Practice",
        "appointment_type": "Follow-up",
        "wait_minutes": 10,
        "scheduled_minutes": 15,
        "status": "Completed",
        "reminder_sent": True,
        "booking_channel": "Online",
    },
    {
        "appointment_id": "APT-1012",
        "booking_date": "2026-09-06",
        "appointment_date": "2026-09-12",
        "department": "Physiotherapy",
        "appointment_type": "Initial",
        "wait_minutes": 30,
        "scheduled_minutes": 45,
        "status": "Missed",
        "reminder_sent": True,
        "booking_channel": "Reception",
    },
]

print("Number of Python records:", len(appointments))
print("First record:", appointments[0])

Number of Python records: 12
First record: {'appointment_id': 'APT-1001', 'booking_date': '2026-09-01', 'appointment_date': '2026-09-05', 'department': 'General Practice', 'appointment_type': 'Initial', 'wait_minutes': 12, 'scheduled_minutes': 20, 'status': 'Completed', 'reminder_sent': True, 'booking_channel': 'Phone'}


### Create a DataFrame

A pandas `DataFrame` is a two-dimensional table with labelled rows and columns. It can contain different data types in different columns.

In [25]:
appointments_df = pd.DataFrame(appointments)
appointments_df

,appointment_id,booking_date,appointment_date,department,appointment_type,wait_minutes,scheduled_minutes,status,reminder_sent,booking_channel
0,APT-1001,2026-09-01,2026-09-05,General Practice,Initial,12.0,20,Completed,True,Phone
1,APT-1002,2026-09-01,2026-09-06,Physiotherapy,Follow-up,25.0,30,Completed,True,Online
2,APT-1003,2026-09-02,2026-09-06,Dermatology,Initial,8.0,20,Completed,False,Phone
3,APT-1004,2026-09-02,2026-09-07,General Practice,Follow-up,NaN,15,Cancelled,True,Online
4,APT-1005,2026-09-03,2026-09-08,Physiotherapy,Initial,40.0,45,Completed,True,Reception
5,APT-1006,2026-09-03,2026-09-08,General Practice,Initial,15.0,20,Missed,False,Phone
6,APT-1007,2026-09-04,2026-09-09,Dermatology,Follow-up,18.0,15,Completed,True,Online
7,APT-1008,2026-09-04,2026-09-10,General Practice,Initial,6.0,20,Completed,True,Reception
8,APT-1009,2026-09-05,2026-09-10,Physiotherapy,Follow-up,22.0,30,Completed,True,Online
9,APT-1010,2026-09-05,2026-09-11,Dermatology,Initial,NaN,20,Cancelled,False,Phone


### Series and DataFrame

Selecting one column produces a `Series`. Selecting a list of columns produces another `DataFrame`.

In [26]:
wait_series = appointments_df["wait_minutes"]
small_table = appointments_df[["appointment_id", "department", "status"]]

print("One column:", type(wait_series))
print("Several columns:", type(small_table))

display(wait_series.head())
display(small_table.head())

One column: <class 'pandas.core.series.Series'>
Several columns: <class 'pandas.core.frame.DataFrame'>


0    12.0
1    25.0
2     8.0
3     NaN
4    40.0
Name: wait_minutes, dtype: float64

,appointment_id,department,status
0,APT-1001,General Practice,Completed
1,APT-1002,Physiotherapy,Completed
2,APT-1003,Dermatology,Completed
3,APT-1004,General Practice,Cancelled
4,APT-1005,Physiotherapy,Completed


### Data dictionary

A data dictionary records what each field is intended to mean. It reduces ambiguity and helps prevent incorrect analysis.

| Column | Intended meaning | Expected role |
|---|---|---|
| `appointment_id` | Unique appointment reference | Identifier |
| `booking_date` | Date the appointment was booked | Date/time |
| `appointment_date` | Scheduled appointment date | Date/time |
| `department` | Administrative service area | Category |
| `appointment_type` | Initial or follow-up booking | Category |
| `wait_minutes` | Recorded administrative wait time | Numeric |
| `scheduled_minutes` | Planned appointment length | Numeric |
| `status` | Completed, cancelled, or missed | Category |
| `reminder_sent` | Whether a reminder was recorded | Boolean |
| `booking_channel` | How the booking was made | Category |

> The data dictionary describes what we **expect**. Inspection tells us what pandas actually received.

## 5. Save and load CSV data

Real analysis usually begins with data stored outside the notebook. To keep this lesson self-contained, we will save the fictional records to a CSV file and read them back.

In [27]:
csv_path = "fictional_appointments_2_1.csv"

# index=False prevents pandas from writing the DataFrame's row index as an extra column.
appointments_df.to_csv(csv_path, index=False)

print(f"Saved {csv_path}")

Saved fictional_appointments_2_1.csv


In [28]:
appointments_df = pd.read_csv(csv_path)

print("Loaded object:", type(appointments_df))
appointments_df.head()

Loaded object: <class 'pandas.core.frame.DataFrame'>


,appointment_id,booking_date,appointment_date,department,appointment_type,wait_minutes,scheduled_minutes,status,reminder_sent,booking_channel
0,APT-1001,2026-09-01,2026-09-05,General Practice,Initial,12.0,20,Completed,True,Phone
1,APT-1002,2026-09-01,2026-09-06,Physiotherapy,Follow-up,25.0,30,Completed,True,Online
2,APT-1003,2026-09-02,2026-09-06,Dermatology,Initial,8.0,20,Completed,False,Phone
3,APT-1004,2026-09-02,2026-09-07,General Practice,Follow-up,NaN,15,Cancelled,True,Online
4,APT-1005,2026-09-03,2026-09-08,Physiotherapy,Initial,40.0,45,Completed,True,Reception


> A CSV file stores plain text. When pandas reads it, pandas must infer each column's data type. That inference may not match the meaning in the data dictionary. Dates, identifiers, missing values, and mixed columns deserve careful attention.

## 6. First inspection: sample rows

`head()` and `tail()` display the first and last rows. `sample()` displays rows from different positions. A fixed `random_state` makes the sample reproducible.

These methods display a subset; they do not change the DataFrame.

In [29]:
appointments_df.head(3)

,appointment_id,booking_date,appointment_date,department,appointment_type,wait_minutes,scheduled_minutes,status,reminder_sent,booking_channel
0,APT-1001,2026-09-01,2026-09-05,General Practice,Initial,12.0,20,Completed,True,Phone
1,APT-1002,2026-09-01,2026-09-06,Physiotherapy,Follow-up,25.0,30,Completed,True,Online
2,APT-1003,2026-09-02,2026-09-06,Dermatology,Initial,8.0,20,Completed,False,Phone


In [30]:
appointments_df.tail(3)

,appointment_id,booking_date,appointment_date,department,appointment_type,wait_minutes,scheduled_minutes,status,reminder_sent,booking_channel
9,APT-1010,2026-09-05,2026-09-11,Dermatology,Initial,NaN,20,Cancelled,False,Phone
10,APT-1011,2026-09-06,2026-09-12,General Practice,Follow-up,10.0,15,Completed,True,Online
11,APT-1012,2026-09-06,2026-09-12,Physiotherapy,Initial,30.0,45,Missed,True,Reception


In [31]:
appointments_df.sample(3, random_state=42)

,appointment_id,booking_date,appointment_date,department,appointment_type,wait_minutes,scheduled_minutes,status,reminder_sent,booking_channel
10,APT-1011,2026-09-06,2026-09-12,General Practice,Follow-up,10.0,15,Completed,True,Online
9,APT-1010,2026-09-05,2026-09-11,Dermatology,Initial,NaN,20,Cancelled,False,Phone
0,APT-1001,2026-09-01,2026-09-05,General Practice,Initial,12.0,20,Completed,True,Phone


### Pause and notice

- Does one row appear to represent one appointment?
- Do the values match the column headings?
- Can you already see any missing values?
- Are there fields whose formatting should be checked later?

Do not rush to explain *why* a pattern exists. First describe what is visible.

## 7. Dimensions and labels

`shape` returns `(number_of_rows, number_of_columns)`. `columns` returns the column labels.

Both are attributes, not methods, so we write them without parentheses.

In [32]:
rows, columns = appointments_df.shape

print("Shape:", appointments_df.shape)
print("Rows:", rows)
print("Columns:", columns)
print("Column labels:")
print(appointments_df.columns.tolist())

Shape: (12, 10)
Rows: 12
Columns: 10
Column labels:
['appointment_id', 'booking_date', 'appointment_date', 'department', 'appointment_type', 'wait_minutes', 'scheduled_minutes', 'status', 'reminder_sent', 'booking_channel']


### Common beginner error

```python
appointments_df.shape()  # Incorrect: shape is not a function
```

Use `appointments_df.shape` instead.

## 8. Data types

`dtypes` reports the type pandas assigned to each column.

Common pandas types include:

- `int64`: whole numbers.
- `float64`: numbers that can contain decimal values or missing values.
- `bool`: `True` or `False`.
- `object`: often text or mixed Python objects.
- `datetime64`: parsed date and time values.

A data type is not the same as a field's analytical role. An appointment ID may be stored as text, but its role is still “identifier.”

In [33]:
appointments_df.dtypes

appointment_id        object
booking_date          object
appointment_date      object
department            object
appointment_type      object
wait_minutes         float64
scheduled_minutes      int64
status                object
reminder_sent           bool
booking_channel       object
dtype: object

### What should stand out?

> The two date columns appear as `object`, not as a date/time type. That means pandas currently treats them like text. The `wait_minutes` column appears as a floating-point type because the column includes missing values. These are not necessarily errors, but they are important findings for the cleaning stage.

We will convert and validate types in Lesson 2.2.

## 9. A compact schema summary

`info()` combines several useful checks: row count, column names, non-null counts, inferred data types, and approximate memory usage.

In [34]:
appointments_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   appointment_id     12 non-null     object 
 1   booking_date       12 non-null     object 
 2   appointment_date   12 non-null     object 
 3   department         12 non-null     object 
 4   appointment_type   12 non-null     object 
 5   wait_minutes       10 non-null     float64
 6   scheduled_minutes  12 non-null     int64  
 7   status             12 non-null     object 
 8   reminder_sent      12 non-null     bool   
 9   booking_channel    12 non-null     object 
dtypes: bool(1), float64(1), int64(1), object(7)
memory usage: 1008.0+ bytes


### Reading `info()`

For each column, compare its **non-null count** with the total number of rows. A smaller count means at least one value is missing.

Answer before moving on:

1. Which column has missing values?
2. How many non-null values does it contain?
3. Which columns were expected to be dates but are currently objects?

## 10. Descriptive summaries

By default, `describe()` summarises numerical columns. It reports count, mean, standard deviation, minimum, quartiles, and maximum.

A summary is a clue, not a conclusion. Always ask whether a calculation is meaningful for the field and whether missing values were excluded.

In [35]:
appointments_df.describe()

,wait_minutes,scheduled_minutes
count,10.000000,12.000000
mean,18.600000,24.583333
std,10.762073,10.757309
min,6.000000,15.000000
25%,10.500000,18.750000
50%,16.500000,20.000000
75%,24.250000,30.000000
max,40.000000,45.000000


### Categorical summaries

For text columns, `describe()` can report the number of non-null values, the number of unique values, the most frequent value, and its frequency.

In [36]:
appointments_df.describe(include="object")

,appointment_id,booking_date,appointment_date,department,appointment_type,status,booking_channel
count,12,12,12,12,12,12,12
unique,12,6,8,3,2,3,3
top,APT-1001,2026-09-01,2026-09-06,General Practice,Initial,Completed,Online
freq,1,2,2,5,7,8,5


### A note about averages

The mean waiting time below is calculated from non-missing values only. The output does not tell us why values are missing or whether the remaining records are representative.

In [37]:
mean_wait = appointments_df["wait_minutes"].mean()
median_wait = appointments_df["wait_minutes"].median()

print(f"Mean recorded wait: {mean_wait:.1f} minutes")
print(f"Median recorded wait: {median_wait:.1f} minutes")

Mean recorded wait: 18.6 minutes
Median recorded wait: 16.5 minutes


## 11. Selecting rows and columns

Selection lets us focus on relevant parts of a table without changing the original data.

- `df["column"]` selects one column as a Series.
- `df[["column_a", "column_b"]]` selects columns as a DataFrame.
- `.loc[]` selects by labels or conditions.
- `.iloc[]` selects by integer position.

In [38]:
# One column: Series
appointments_df["department"].head()

0    General Practice
1       Physiotherapy
2         Dermatology
3    General Practice
4       Physiotherapy
Name: department, dtype: object

In [39]:
# Several columns: DataFrame
appointments_df[["appointment_id", "department", "status"]].head()

,appointment_id,department,status
0,APT-1001,General Practice,Completed
1,APT-1002,Physiotherapy,Completed
2,APT-1003,Dermatology,Completed
3,APT-1004,General Practice,Cancelled
4,APT-1005,Physiotherapy,Completed


In [40]:
# First two rows and first four columns, selected by position
appointments_df.iloc[0:2, 0:4]

,appointment_id,booking_date,appointment_date,department
0,APT-1001,2026-09-01,2026-09-05,General Practice
1,APT-1002,2026-09-01,2026-09-06,Physiotherapy


In [41]:
# Completed appointments, selected with a Boolean condition
completed_appointments = appointments_df.loc[
    appointments_df["status"] == "Completed",
    ["appointment_id", "department", "wait_minutes"],
]

completed_appointments

,appointment_id,department,wait_minutes
0,APT-1001,General Practice,12.0
1,APT-1002,Physiotherapy,25.0
2,APT-1003,Dermatology,8.0
4,APT-1005,Physiotherapy,40.0
6,APT-1007,Dermatology,18.0
7,APT-1008,General Practice,6.0
8,APT-1009,Physiotherapy,22.0
10,APT-1011,General Practice,10.0


## 12. Quick categorical counts

`value_counts()` reports how often each distinct value appears. It is a fast first check for categorical columns.

In [42]:
appointments_df["status"].value_counts()

status
Completed    8
Cancelled    2
Missed       2
Name: count, dtype: int64

In [43]:
appointments_df["department"].value_counts()

department
General Practice    5
Physiotherapy       4
Dermatology         3
Name: count, dtype: int64

> We can observe that some categories occur more often in this small fictional file. We cannot yet claim that a department is generally busier, because the dataset covers only a few invented records and may not represent a longer period.

## 13. Missing values and unique identifiers

We are not cleaning yet, but we should record obvious quality questions.

- `isna()` marks missing values as `True`.
- `sum()` counts the `True` values in each column.
- `duplicated()` helps check whether an intended identifier appears more than once.

In [44]:
missing_counts = appointments_df.isna().sum()
missing_counts

appointment_id       0
booking_date         0
appointment_date     0
department           0
appointment_type     0
wait_minutes         2
scheduled_minutes    0
status               0
reminder_sent        0
booking_channel      0
dtype: int64

In [45]:
duplicate_id_count = appointments_df["appointment_id"].duplicated().sum()

print("Duplicate appointment IDs:", duplicate_id_count)
print("Unique appointment IDs:", appointments_df["appointment_id"].nunique())
print("Total rows:", len(appointments_df))

Duplicate appointment IDs: 0
Unique appointment IDs: 12
Total rows: 12


## 14. Build a data inventory

A data inventory gives us a structured starting point for later cleaning and analysis. This inventory records each column's inferred type, missing-value count, number of distinct non-missing values, and an example value.

In [46]:
def first_non_missing(series):
    """Return the first non-missing value in a pandas Series."""
    non_missing = series.dropna()
    return non_missing.iloc[0] if not non_missing.empty else None


data_inventory = pd.DataFrame({
    "inferred_dtype": appointments_df.dtypes.astype(str),
    "missing_values": appointments_df.isna().sum(),
    "unique_non_missing": appointments_df.nunique(dropna=True),
    "example_value": [
        first_non_missing(appointments_df[column])
        for column in appointments_df.columns
    ],
})

data_inventory.index.name = "column"
data_inventory

,inferred_dtype,missing_values,unique_non_missing,example_value
column,,,,
appointment_id,object,0,12,APT-1001
booking_date,object,0,6,2026-09-01
appointment_date,object,0,8,2026-09-05
department,object,0,3,General Practice
appointment_type,object,0,2,Initial
wait_minutes,float64,2,10,12.0
scheduled_minutes,int64,0,4,20
status,object,0,3,Completed
reminder_sent,bool,0,2,True


### Interpret the inventory

For each column, ask:

1. Does the inferred data type match the intended meaning?
2. Are missing values plausible, or do they require investigation?
3. Should values be unique?
4. Is the example value in an expected format?
5. Is the field an identifier, numerical measure, category, date/time, or Boolean flag?

This is the bridge from “I loaded a file” to “I understand what must be checked before analysis.”

## 15. Observation versus explanation

EDA requires careful language.

| Type | Example |
|---|---|
| Observation | “Two fictional records have a missing waiting time.” |
| Possible explanation | “The appointments may have been cancelled before a wait was recorded.” |
| Unsupported claim | “Cancelled appointments always have no waiting time.” |

An observation follows directly from the data. An explanation is a hypothesis that needs evidence. A small dataset rarely justifies words such as “always,” “causes,” or “proves.”

In [47]:
# Inspect the records with missing waiting times.
appointments_df.loc[
    appointments_df["wait_minutes"].isna(),
    ["appointment_id", "status", "wait_minutes"],
]

column,appointment_id,status,wait_minutes
3,APT-1004,Cancelled,NaN
9,APT-1010,Cancelled,NaN


### Write carefully

Complete these statements:

- **Observation:** I can see that...
- **Question:** I would like to investigate whether...
- **Information needed:** To investigate this, I would need...

Avoid turning a pattern in this small file into a general claim about clinics, staff, or patients.

## 16. Code-along challenge: inspect an unfamiliar DataFrame

Use the same inspection sequence whenever you receive a new table:

1. Preview rows.
2. Check dimensions.
3. List columns.
4. Inspect data types and non-null counts.
5. Produce numerical and categorical summaries.
6. Count missing values.
7. Check the intended identifier.

Complete the cell below. The comments show the required steps.

In [48]:
# Your turn: replace each comment with working code.

# 1. Display the first four rows.

# 2. Print the number of rows and columns.

# 3. Display the column names.

# 4. Display the inferred data types.

# 5. Run the compact schema summary.

# 6. Display numerical descriptive statistics.

# 7. Count missing values in every column.

# 8. Count duplicated appointment IDs.

<details>
<summary><strong>Reveal one possible solution</strong></summary>

Run the next code cell after attempting the challenge yourself. There are often several correct ways to display the same information.
</details>

In [49]:
display(appointments_df.head(4))
print("Shape:", appointments_df.shape)
print("Columns:", appointments_df.columns.tolist())
display(appointments_df.dtypes)
appointments_df.info()
display(appointments_df.describe())
display(appointments_df.isna().sum())
print(
    "Duplicated appointment IDs:",
    appointments_df["appointment_id"].duplicated().sum(),
)

column,appointment_id,booking_date,appointment_date,department,appointment_type,wait_minutes,scheduled_minutes,status,reminder_sent,booking_channel
0,APT-1001,2026-09-01,2026-09-05,General Practice,Initial,12.0,20,Completed,True,Phone
1,APT-1002,2026-09-01,2026-09-06,Physiotherapy,Follow-up,25.0,30,Completed,True,Online
2,APT-1003,2026-09-02,2026-09-06,Dermatology,Initial,8.0,20,Completed,False,Phone
3,APT-1004,2026-09-02,2026-09-07,General Practice,Follow-up,NaN,15,Cancelled,True,Online


Shape: (12, 10)
Columns: ['appointment_id', 'booking_date', 'appointment_date', 'department', 'appointment_type', 'wait_minutes', 'scheduled_minutes', 'status', 'reminder_sent', 'booking_channel']


column
appointment_id        object
booking_date          object
appointment_date      object
department            object
appointment_type      object
wait_minutes         float64
scheduled_minutes      int64
status                object
reminder_sent           bool
booking_channel       object
dtype: object

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   appointment_id     12 non-null     object 
 1   booking_date       12 non-null     object 
 2   appointment_date   12 non-null     object 
 3   department         12 non-null     object 
 4   appointment_type   12 non-null     object 
 5   wait_minutes       10 non-null     float64
 6   scheduled_minutes  12 non-null     int64  
 7   status             12 non-null     object 
 8   reminder_sent      12 non-null     bool   
 9   booking_channel    12 non-null     object 
dtypes: bool(1), float64(1), int64(1), object(7)
memory usage: 1008.0+ bytes


column,wait_minutes,scheduled_minutes
count,10.000000,12.000000
mean,18.600000,24.583333
std,10.762073,10.757309
min,6.000000,15.000000
25%,10.500000,18.750000
50%,16.500000,20.000000
75%,24.250000,30.000000
max,40.000000,45.000000


column
appointment_id       0
booking_date         0
appointment_date     0
department           0
appointment_type     0
wait_minutes         2
scheduled_minutes    0
status               0
reminder_sent        0
booking_channel      0
dtype: int64

Duplicated appointment IDs: 0


## 17. Independent practice: make a mini data inventory

Create a concise Markdown report using results from the notebook. Include:

- The unit of observation.
- The dataset shape.
- The intended identifier.
- Two numerical fields.
- Three categorical fields.
- Fields containing missing values.
- Expected date fields that pandas currently stores as text.
- Two questions you would ask the person who supplied the data.

### Your report

- **Unit of observation:** 
- **Shape:** 
- **Identifier:** 
- **Numerical fields:** 
- **Categorical fields:** 
- **Missing fields:** 
- **Date fields currently stored as text:** 
- **Question 1:** 
- **Question 2:**

### Extension: write a reusable inspection function

Functions help us apply the same inspection steps consistently to future datasets. Complete or modify the function below.

In [50]:
def inspect_dataframe(df, id_column=None):
    """Print a beginner-friendly first inspection of a DataFrame."""
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("\nData types:")
    print(df.dtypes)
    print("\nMissing values:")
    print(df.isna().sum())

    if id_column is not None:
        print(f"\nDuplicate values in {id_column}:", df[id_column].duplicated().sum())


inspect_dataframe(appointments_df, id_column="appointment_id")

Shape: (12, 10)
Columns: ['appointment_id', 'booking_date', 'appointment_date', 'department', 'appointment_type', 'wait_minutes', 'scheduled_minutes', 'status', 'reminder_sent', 'booking_channel']

Data types:
column
appointment_id        object
booking_date          object
appointment_date      object
department            object
appointment_type      object
wait_minutes         float64
scheduled_minutes      int64
status                object
reminder_sent           bool
booking_channel       object
dtype: object

Missing values:
column
appointment_id       0
booking_date         0
appointment_date     0
department           0
appointment_type     0
wait_minutes         2
scheduled_minutes    0
status               0
reminder_sent        0
booking_channel      0
dtype: int64

Duplicate values in appointment_id: 0


### Stretch task

Extend `inspect_dataframe()` so it also:

- Displays the first three rows.
- Prints the number of distinct values in every column.
- Warns when the requested ID column does not exist.
- Returns a DataFrame inventory instead of only printing results.

Start with one improvement at a time.

In [51]:
# Write your extended function here.
# Keeping this cell comment-only means the notebook still runs from top to bottom.

## 18. Test your knowledge!

Answer without running new code first.

1. What does one row in `appointments_df` represent?
2. What is the difference between a pandas `Series` and a `DataFrame`?
3. If a DataFrame has shape `(12, 10)`, how many observations and columns does it contain?
4. Which inspection method shows non-null counts and data types together?
5. Why are `booking_date` and `appointment_date` worth checking?
6. Why is `appointment_id` not a numerical measurement?
7. What does vectorization allow us to do?
8. What is the difference between an observation and an explanation?
9. Which column in the fictional dataset contains missing values?
10. Name three checks you should perform before calculating a result from unfamiliar data.

<details>
<summary><strong>Check your answers</strong></summary>

1. One scheduled appointment.
2. A Series is one-dimensional and labelled; a DataFrame is a two-dimensional table with labelled rows and columns.
3. 12 observations and 10 columns.
4. `info()`.
5. pandas currently reads them as text (`object`), so date operations would require validation and conversion.
6. It labels a record; arithmetic on it would not have a meaningful interpretation.
7. Apply an operation across an array without writing an explicit Python loop for each value.
8. An observation is directly supported by the data; an explanation is a hypothesis about why it occurred.
9. `wait_minutes`.
10. Examples include previewing rows, checking shape, reviewing columns and types, counting missing values, and checking identifier uniqueness.

</details>

## 19. Lesson recap

You have completed the first stage of systematic EDA: **understanding the schema**.

You can now:

- Move from Python lists and dictionaries to NumPy and pandas.
- Use vectorized numerical operations.
- distinguish a Series from a DataFrame.
- Load tabular data from CSV.
- Inspect samples, dimensions, labels, types, non-null counts, and descriptive summaries.
- Build a data inventory.
- Describe evidence without inventing an explanation.

In Lesson 2.2, we will preserve the raw data, create a cleaned copy, convert dates and numbers, standardise categories, investigate missing values, and document every cleaning decision.

## Further reading

- [NumPy: What is NumPy?](https://numpy.org/doc/stable/user/whatisnumpy.html)
- [pandas DataFrame reference](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
- [pandas essential basic functionality](https://pandas.pydata.org/docs/user_guide/basics.html)

These references are useful when you want to confirm the exact behaviour of an attribute or method.